# Imports

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import copy
import json
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.imagenet import IMAGENET_MEAN, IMAGENET_STD, to_batch
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.eval import (MEAN_KEY, bilinear_argmax, evaluate, evaluate_predictions,
                                      load_baseline_model, load_val_sets, lraspp_outputs, results_table,
                                      run_model)
from uavloc.segmentation.latency import cpu_latency, standard_tile
from uavloc.segmentation.refine.boundary_head import (LossWeights, band_counts, band_scores, boundary_probabilities,
                                                      boundary_target, freeze_batchnorm, load_boundary_model,
                                                      multitask_loss, multitask_outputs, optimizer, watershed_fusion)
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

warnings.filterwarnings("ignore")

# Params

In [ ]:
EXP = "015-boundary-head"
NOTE = ""
LOG_RUN = True

WITH_HEAD = True
SEED = 42
INIT_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
COMPARE_CHECKPOINT = ""          # figure only: a control run's checkpoint in artifacts/, same seed

STEPS = 1500
LOSS_LOG_EVERY = 50
SAT_CROP_PX, SAT_BATCH = 512, 4
RARE_CROP_P = 0.3
LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
BOUNDARY_WEIGHT = 1.0
BOUNDARY_RADIUS_PX = 1

FUSION_THRESHOLD = 0.5
DIAGNOSTIC_THRESHOLDS = [0.3, 0.7]
LATENCY_BUDGET_RATIO = 0.35
LATENCY_ROUNDS = 3

HOLDOUT_VILLAGE = "bushcha-50p2422-25p8305"
STRIP_VILLAGE = "birky-49p9105-23p9192"
STRIP_ROWS = 640
GAP_PX = 64
DEVICE = "mps"

REFERENCE_HARNESS = {"miou": 0.476, "biou": 0.123}
REFERENCE_TOLERANCE = 0.003
LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
ZOOM_BOXES_PX = {"birky-strip-google": [120, 1400, 376, 1912], "bushcha": [1024, 1024, 1280, 1536]}
TARGET_SAMPLE_SEED = 7

In [ ]:
set_seed(SEED)
torch.manual_seed(SEED)
ARM = "head" if WITH_HEAD else "control"
TAG = f"{ARM}-seed{SEED}"
ARTIFACTS = experiment_dir(EXP) / "artifacts"
FIGURES = experiment_dir(EXP) / "figures"
TAG

# Data

In [ ]:
DERIVED = REPO_ROOT / "data" / "derived"
sat = vs.load_satellite(DERIVED / "ua-village-annotated", DERIVED)
vs.ban_holdouts(sat, HOLDOUT_VILLAGE, STRIP_VILLAGE, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
sampler = vs.SatelliteSampler(sat_train, SAT_CROP_PX, RARE_CROP_P)
[(s.loc.split("-")[0], s.date) for s in sat_train]

In [ ]:
val_sets = load_val_sets()
{name: len(images) for name, images in val_sets.items()}

# Model

In [ ]:
model = load_boundary_model(REPO_ROOT / INIT_CHECKPOINT, DEVICE, WITH_HEAD)
reference = load_baseline_model(DEVICE)
check_x = torch.from_numpy(vs.batch_of(sampler, np.random.default_rng(SEED), 1)[0].numpy()).to(DEVICE)
with torch.no_grad():
    init_diff = (model.eval().coarse_logits(check_x) - reference.coarse_logits(check_x)).abs().max().item()
assert init_diff == 0, init_diff
init_diff

In [ ]:
opt = optimizer(model, LR, HEAD_LR, WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in opt.param_groups], total_steps=STEPS,
                                            pct_start=WARMUP_FRAC)
weights = LossWeights(BOUNDARY_WEIGHT, BOUNDARY_RADIUS_PX)

# Run

In [ ]:
rng = np.random.default_rng(SEED)
history, window, t0 = [], [], time.time()
for step in range(1, STEPS + 1):
    model.train()
    freeze_batchnorm(model)
    x, y = vs.batch_of(sampler, rng, SAT_BATCH)
    loss, parts = multitask_loss(model, x.to(DEVICE), y.to(DEVICE), weights)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    window.append(parts)
    if step % LOSS_LOG_EVERY == 0:
        history.append({"step": step, "minutes": (time.time() - t0) / 60, **pd.DataFrame(window).mean().to_dict()})
        print(history[-1], flush=True)
        window = []
train_minutes = (time.time() - t0) / 60
model.eval()
checkpoint = ARTIFACTS / f"model-{TAG}.pt"
torch.save(model.state_dict(), checkpoint)
train_minutes

In [ ]:
t0 = time.time()
reference_results = evaluate(bilinear_argmax, run_model(reference, val_sets, DEVICE))
cases = run_model(model, val_sets, DEVICE)
argmax_results = evaluate(bilinear_argmax, cases)
results_table(argmax_results).round(4)

In [ ]:
reference_check = {k: reference_results[MEAN_KEY][k] - v for k, v in REFERENCE_HARNESS.items()}
assert all(abs(d) <= REFERENCE_TOLERANCE for d in reference_check.values()), reference_check
reference_check

In [ ]:
def fuse_all(cases: dict, boundary_maps: dict, threshold: float) -> dict[str, list[np.ndarray]]:
    """Watershed fusion of every case's semantic probabilities with its boundary map."""
    return {name: [watershed_fusion(c.output.probs, b, threshold) for c, b in zip(items, boundary_maps[name])]
            for name, items in cases.items()}

In [ ]:
def gt_bands(cases: dict, radius_px: int) -> dict[str, list[np.ndarray]]:
    """The training target band of each val label, as a 0/1 boundary map (oracle input to the fusion)."""
    return {name: [boundary_target(torch.from_numpy(c.image.label.astype(np.int64))[None], radius_px)[0][0].numpy()
                   for c in items] for name, items in cases.items()}

In [ ]:
def band_f1(boundary_maps: dict, cases: dict, radius_px: int, threshold: float) -> dict[str, dict]:
    """Pixel precision / recall / F1 of the predicted band per val set, and their unweighted mean."""
    scores = {name: band_scores(sum(band_counts(b, c.image.label, radius_px, threshold)
                                    for b, c in zip(boundary_maps[name], items)))
              for name, items in cases.items()}
    scores[MEAN_KEY] = pd.DataFrame(scores).T.mean().to_dict()
    return scores

In [ ]:
oracle_results = evaluate_predictions(fuse_all(cases, gt_bands(cases, BOUNDARY_RADIUS_PX), FUSION_THRESHOLD), cases)
if WITH_HEAD:
    boundary_maps = {name: [boundary_probabilities(model, c.image.rgb, DEVICE) for c in items]
                     for name, items in cases.items()}
    fused_results = {t: evaluate_predictions(fuse_all(cases, boundary_maps, t), cases)
                     for t in [FUSION_THRESHOLD, *DIAGNOSTIC_THRESHOLDS]}
    band_quality = band_f1(boundary_maps, cases, BOUNDARY_RADIUS_PX, FUSION_THRESHOLD)
eval_minutes = (time.time() - t0) / 60
eval_minutes

In [ ]:
def prefixed(prefix: str, results: dict) -> dict[str, float]:
    """Mean scalars as `<prefix>_<metric>_mean`, plus Boundary IoU per val set."""
    out = {f"{prefix}_{k}_mean": v for k, v in results[MEAN_KEY].items()}
    out.update({f"{prefix}_biou_{name}": results[name]["biou"] for name in results if name != MEAN_KEY})
    return out

In [ ]:
metrics = {"biou_mean": argmax_results[MEAN_KEY]["biou"], "miou_mean": argmax_results[MEAN_KEY]["miou"],
           **prefixed("argmax", argmax_results), **prefixed("ref", reference_results),
           **prefixed("oracle_fused", oracle_results), "train_minutes": train_minutes, "eval_minutes": eval_minutes,
           "final_seg_loss": history[-1]["seg"]}
if WITH_HEAD:
    for t, r in fused_results.items():
        metrics.update(prefixed(f"fused_t{t:g}", r))
    metrics.update({f"band_{k}_mean": v for k, v in band_quality[MEAN_KEY].items()})
    metrics["final_boundary_loss"] = history[-1]["boundary"]
pd.Series(metrics).filter(like="_mean").round(4)

In [ ]:
def interleaved_latency(fns: dict, rounds: int) -> pd.DataFrame:
    """Median ms of each fn, timed in alternating rounds so a shared machine's load hits all of them alike."""
    rows = [{"round": r, "name": name, "median_ms": cpu_latency(fn).median_ms}
            for r in range(rounds) for name, fn in fns.items()]
    return pd.DataFrame(rows).pivot(index="round", columns="name", values="median_ms")

In [ ]:
latency_image = val_sets[LATENCY_SET][0]
latency_tile = standard_tile(latency_image.rgb, *LATENCY_TOP_LEFT_PX)
cpu_model = copy.deepcopy(model).cpu().eval()
cpu_reference = copy.deepcopy(reference).cpu().eval()
timed = {"lraspp": lambda: lraspp_outputs(cpu_reference, latency_tile, "cpu")}
if WITH_HEAD:
    tile_output, tile_boundary = multitask_outputs(cpu_model, latency_tile, "cpu")
    timed["multitask"] = lambda: multitask_outputs(cpu_model, latency_tile, "cpu")
    tile_features = cpu_model.features(to_batch(latency_tile, "cpu"))
    timed["head"] = lambda: cpu_model.head(tile_features["stride4"], tile_features["low"])
    timed["fusion"] = lambda: watershed_fusion(tile_output.probs, tile_boundary, FUSION_THRESHOLD)
latency = interleaved_latency(timed, LATENCY_ROUNDS)
latency.round(1)

In [ ]:
metrics["latency_lraspp_ms"] = float(latency["lraspp"].median())
if WITH_HEAD:
    overhead = (latency["head"] + latency["fusion"]) / latency["lraspp"]
    metrics.update({"latency_multitask_ms": float(latency["multitask"].median()),
                    "latency_head_ms": float(latency["head"].median()),
                    "latency_fusion_ms": float(latency["fusion"].median()),
                    "latency_overhead_ratio": float(overhead.median())})
    metrics["latency_within_budget"] = metrics["latency_overhead_ratio"] <= LATENCY_BUDGET_RATIO
{k: v for k, v in metrics.items() if k.startswith("latency")}

# Plots

In [ ]:
def colour(label: np.ndarray) -> np.ndarray:
    out = np.full((*label.shape, 3), 255, np.uint8)
    ok = label != IGNORE_LABEL
    out[ok] = SEG_PALETTE[label[ok]]
    return out

In [ ]:
hist = pd.DataFrame(history)
fig_loss, ax = plt.subplots(figsize=(8, 3.5))
for col in [c for c in ("seg", "boundary") if c in hist]:
    ax.plot(hist["step"], hist[col], label=f"{col} loss")
ax.set_xlabel("step"); ax.set_title(f"training loss, {TAG}"); ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
sample_x, sample_y = vs.batch_of(sampler, np.random.default_rng(TARGET_SAMPLE_SEED), 1)
sample_band, sample_valid = boundary_target(sample_y, BOUNDARY_RADIUS_PX)
sample_rgb = np.clip(sample_x[0].numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN, 0, 1)
fig_target, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (img, title) in zip(axes, [(sample_rgb, "training crop"), (colour(sample_y[0].numpy()), "label"),
                                   (sample_band[0] * sample_valid[0], "boundary target (scored band)")]):
    ax.imshow(img, interpolation="nearest"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); fig_target.savefig(FIGURES / "boundary-target.png", dpi=90, bbox_inches="tight"); plt.show()

In [ ]:
def zoom_panels(name: str, box: list[int]) -> list[tuple[np.ndarray, str]]:
    """RGB | label | original LR-ASPP | [control fine-tune] | this run | [fused | boundary prob] for one crop."""
    top, left, bottom, right = box
    zoom, case = np.s_[top:bottom, left:right], cases[name][0]
    panels = [(case.image.rgb[zoom], f"{name} RGB"), (colour(case.image.label[zoom]), "label"),
              (colour(reference_preds[name][zoom]), "LR-ASPP (original)")]
    if compare_preds:
        panels.append((colour(compare_preds[name][zoom]), "control fine-tune"))
    panels.append((colour(bilinear_argmax(case.output, case.image.rgb)[zoom]), f"{ARM} fine-tune, argmax"))
    if WITH_HEAD:
        fused = watershed_fusion(case.output.probs, boundary_maps[name][0], FUSION_THRESHOLD)
        panels += [(colour(fused[zoom]), f"{ARM} + watershed fusion"), (boundary_maps[name][0][zoom], "boundary prob")]
    return panels

In [ ]:
zoom_images = {name: val_sets[name][:1] for name in ZOOM_BOXES_PX}
reference_preds = {name: bilinear_argmax(c.output, c.image.rgb)
                   for name, (c,) in run_model(reference, zoom_images, DEVICE).items()}
compare_preds = {}
if COMPARE_CHECKPOINT:
    compare_model = load_boundary_model(experiment_dir(EXP) / COMPARE_CHECKPOINT, DEVICE, with_head=False).eval()
    compare_preds = {name: bilinear_argmax(c.output, c.image.rgb)
                     for name, (c,) in run_model(compare_model, zoom_images, DEVICE).items()}

In [ ]:
rows = [zoom_panels(name, box) for name, box in ZOOM_BOXES_PX.items()]
fig_zoom, axes = plt.subplots(len(rows[0]), len(rows), figsize=(9 * len(rows), 3.2 * len(rows[0])), squeeze=False)
for col, panels in enumerate(rows):
    for row, (img, title) in enumerate(panels):
        axes[row, col].imshow(img, interpolation="nearest", cmap="magma", vmin=0, vmax=1)
        axes[row, col].set_title(title); axes[row, col].axis("off")
fig_zoom.legend([plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))], CLASSES,
                loc="lower center", ncol=len(CLASSES))
plt.tight_layout(rect=(0, .02, 1, 1)); fig_zoom.savefig(FIGURES / f"zoom-{TAG}.png", dpi=90, bbox_inches="tight")
plt.show()

In [ ]:
RUN_CONFIG = {
    "arm": ARM, "with_head": WITH_HEAD, "init_checkpoint": INIT_CHECKPOINT, "steps": STEPS,
    "sat_crop_px": SAT_CROP_PX, "sat_batch": SAT_BATCH, "rare_crop_p": RARE_CROP_P, "lr": LR, "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY, "warmup_frac": WARMUP_FRAC, "boundary_weight": BOUNDARY_WEIGHT,
    "boundary_radius_px": BOUNDARY_RADIUS_PX, "batchnorm": "frozen-satellite", "loss": "ce+dice (+ balanced bce)",
    "fusion": "watershed", "fusion_threshold": FUSION_THRESHOLD, "holdout": [HOLDOUT_VILLAGE, STRIP_VILLAGE],
    "strip_rows": STRIP_ROWS, "gap_px": GAP_PX, "val_sets": list(val_sets), "boundary_iou_px": 4,
}

In [ ]:
per_class = {"argmax": argmax_results, "ref": reference_results, "oracle_fused": oracle_results,
             **({f"fused_t{t:g}": r for t, r in fused_results.items()} if WITH_HEAD else {})}
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_loss, fig_zoom], seed=SEED,
              checkpoint=str(checkpoint.relative_to(REPO_ROOT)), device=DEVICE) if LOG_RUN else {"run": "dry"}
(ARTIFACTS / f"results-{run['run']}.json").write_text(json.dumps(
    {"per_class": per_class, "history": history, "latency_ms_by_round": latency.to_dict(),
     **({"band_quality": band_quality} if WITH_HEAD else {})}, indent=1, default=str))
run["run"]